#Exploratory Data Analysis: Revenue & Cohorts

In [33]:
!pip install duckdb -q
!pip install -U kaleido
import duckdb

con = duckdb.connect()

path = "all my for the project"

files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}
for table_name, file_name in files.items():
    con.execute(f"""
        create table if not exists {table_name} as
        select * from read_csv_auto(
            '{path}/{file_name}',
            ignore_errors=true,
            null_padding=true
        )
    """)

print("Data base is created!")
display(con.execute("show tables").df())

import plotly.io as pio
pio.renderers.default = "colab"

Data base is created!


,name
0,category_translation
1,customers
2,geolocation
3,order_items
4,order_payments
5,order_reviews
6,orders
7,products
8,sellers


In [34]:
query_monthly = """
    select
        strftime(order_purchase_timestamp, '%m-%Y') as month,
        count(distinct order_id) as total_orders,
        round(sum(price), 2) as total_revenue,
        round(avg(price), 2) as avg_order_value
    from orders
    join order_items using (order_id)
    where order_status = 'delivered'
    group by strftime(order_purchase_timestamp, '%m-%Y')
    order by min(order_purchase_timestamp)
"""

df_monthly = con.execute(query_monthly).df()
display(df_monthly.head(10))

,month,total_orders,total_revenue,avg_order_value
0,10-2016,53,7778.42,129.64
1,01-2017,166,25236.28,124.93
2,02-2017,354,52823.32,126.37
3,03-2017,554,73587.47,117.55
4,04-2017,529,75176.34,127.42
5,05-2017,830,121320.12,128.52
6,06-2017,686,95170.36,122.48
7,07-2017,843,119973.76,122.42
8,08-2017,996,132147.13,118.09
9,09-2017,922,142091.10,134.56


In [35]:
query_retention = """
    with first_purchase as (
        select
            c.customer_unique_id,
            min(date_trunc('month', o.order_purchase_timestamp)) as cohort_month
        from orders o
        join customers c using (customer_id)
        where o.order_status = 'delivered'
        group by 1
    ),
    user_activities as (
        select
            c.customer_unique_id,
            fp.cohort_month,
            date_trunc('month', o.order_purchase_timestamp) as activity_month,
            (date_part('year', date_trunc('month', o.order_purchase_timestamp)) - date_part('year', fp.cohort_month)) * 12 +
            (date_part('month', date_trunc('month', o.order_purchase_timestamp)) - date_part('month', fp.cohort_month)) as month_number
        from orders o
        join customers c using (customer_id)
        join first_purchase fp using (customer_unique_id)
        where o.order_status = 'delivered'
    )
    select
        strftime(cohort_month, '%Y-%m') as cohort,
        month_number,
        count(distinct customer_unique_id) as active_users
    from user_activities
    group by 1, 2
    order by 1, 2
"""

df_retention = con.execute(query_retention).df()
display(df_retention.head(15))

,cohort,month_number,active_users
0,2016-09,0,1
1,2016-10,0,36
2,2016-12,0,1
3,2017-01,0,77
4,2017-02,0,183
5,2017-02,11,1
6,2017-03,0,265
7,2017-04,0,252
8,2017-05,0,408
9,2017-05,1,1


In [36]:
import plotly.express as px
import plotly.graph_objects as go


In [37]:

fig_revenue = px.line(
    df_monthly,
    x='month',
    y='total_revenue',
    title='monthly revenue trend ($)',
    labels={'month': 'month', 'total_revenue': 'revenue ($)'},
    markers=True
)
fig_revenue.update_traces(line_color='#2b5c8f', line_width=3)
fig_revenue.show()


Fact: Mounthly reveue grew steadily from late 2016, reaching a peak og around 326k in March 2018, followed by a downward trend to 262k

Root Cause: Customers might be losing interest over time, or the platform lacks engagement triggers for repeat purchases.

Recommendation: We should introduce special offers, promotional campaigns, or a loyalty/bonus point system to re-engage users. Also, it is necessary to check which specific product categories experienced the highest demand drop

In [38]:
retention_pivot = df_retention.pivot(
    index="cohort", columns="month_number", values="active_users"
)

cohort_sizes = retention_pivot[0]
retention_pct = retention_pivot.divide(cohort_sizes, axis=0) * 100
retention_pct_clean = retention_pct.drop(columns=[0])

fig_heatmap = px.imshow(
    retention_pct_clean,
    labels=dict(x="month number (after purchase)", y="cohort", color="retention (%)"),
    x=retention_pct_clean.columns,
    y=retention_pct_clean.index,
    color_continuous_scale="YlGnBu",
    title="cohort retention rate (%)",
    text_auto=".2f",
)

fig_heatmap.update_traces(
    xgap=3, ygap=3
)

fig_heatmap.update_layout(
    height=800,
    xaxis=dict(dtick=1),
    paper_bgcolor="#f8f9fa",
    plot_bgcolor="#f8f9fa",
)

fig_heatmap.show()

Fact: The cohort retention heatmap shows that repeat purchase rates are extremely low. The highest spikes appear in early cohorts (like Apr 2017), but for the vast majority of months, retention drops close to 0%.

Root Cause: The platform functions as a one-time purchase marketplace. Once customers buy an item (often large goods like furniture), they have no reason to return.

Recommendation: Since users do not return organically, the business needs automated email retention campaigns and push notifications right after the first delivery to encourage a second purchase within the first 30 days.

#Product Category & Logistics Cost Analysis

In [39]:
query_categories = """
    select
        coalesce(t.product_category_name_english, p.product_category_name) as category_english,
        count(distinct oi.order_id) as total_orders,
        round(sum(oi.price), 2) as total_revenue,
        round(avg(oi.freight_value), 2) as avg_freight_cost,
        round(avg(oi.freight_value / oi.price) * 100, 1) as freight_to_price_pct
    from order_items oi
    join products p using (product_id)
    left join category_translation t using (product_category_name)
    group by 1
    order by total_orders desc
    limit 10
"""

df_categories = con.execute(query_categories).df()
display(df_categories)


,category_english,total_orders,total_revenue,avg_freight_cost,freight_to_price_pct
0,bed_bath_table,4466,488201.64,18.38,29.0
1,health_beauty,4080,570298.33,19.07,28.4
2,sports_leisure,3622,455793.86,19.37,28.9
3,computers_accessories,3184,429780.37,18.87,29.4
4,furniture_decor,3040,349457.20,20.90,34.3
5,housewares,2819,301423.80,20.83,39.5
6,watches_gifts,2731,588163.30,16.60,16.9
7,telephony,1983,158434.36,15.77,51.3
8,toys,1831,223426.85,18.75,26.0
9,auto,1774,268801.05,22.22,32.7


In [40]:

import plotly.express as px

fig_categories = px.bar(
    df_categories.sort_values('total_revenue', ascending=True),
    x='total_revenue',
    y='category_english',
    orientation='h',
    title='Top 10 Product Categories by Revenue ($)',
    labels={'total_revenue': 'Total Revenue ($)', 'category_english': 'Category'},
    text_auto='$:,.0f'
)

fig_categories.update_layout(
    height=500,
    xaxis_title="Total Revenue ($)",
    yaxis_title="",
    paper_bgcolor="#f8f9fa",
    plot_bgcolor="#f8f9fa"
)

fig_categories.show()

Fact: Revenue is led by health_beauty (1.25M) and watches_gifts (1.20M). However, bed_bath_table ranks highest in total order volume (9,417 orders). Lower-ranking categories like telephony show a very high freight-to-price ratio (50.6%).

Root Cause: High-ticket items (watches) drive revenue through higher average order value, while lower-priced categories rely on volume. Categories like telephony underperform because shipping costs account for over half of the product price, discouraging buyers.

Recommendation: Introduce bundled shipping discounts for low-cost categories (like phone accessories) to offset shipping fees and boost order volume.


# Logistics Delays & Customer Satisfaction Impact

In [41]:
query_delivery = """
    select
        case
            when order_delivered_customer_date > order_estimated_delivery_date then 'Delayed'
            else 'On Time'
        end as delivery_status,
        count(distinct order_id) as total_orders,
        round(avg(review_score), 2) as avg_review_score,
        round(count(distinct order_id) * 100.0 / sum(count(distinct order_id)) over(), 1) as order_share_pct
    from orders
    join order_reviews using (order_id)
    where order_status = 'delivered'
      and order_delivered_customer_date is not null
    group by 1
"""

df_delivery = con.execute(query_delivery).df()
display(df_delivery)

,delivery_status,total_orders,avg_review_score,order_share_pct
0,Delayed,2089,2.58,7.8
1,On Time,24548,4.30,92.2


In [42]:
import plotly.express as px

fig_delivery = px.bar(
    df_delivery,
    x='delivery_status',
    y='avg_review_score',
    title='Average Review Score: On Time vs Delayed',
    labels={'delivery_status': 'Delivery Status', 'avg_review_score': 'Average Review Score'},
    text_auto='.2f',
    color='delivery_status',
    color_discrete_map={'On Time': '#2b5c8f', 'Delayed': '#d9534f'}
)

fig_delivery.update_layout(
    height=450,
    paper_bgcolor="#f8f9fa",
    plot_bgcolor="#f8f9fa",
    showlegend=False
)

fig_delivery.show()

Fact: On-time deliveries receive an average customer satisfaction score of 4.29 out of 5, whereas delayed deliveries suffer a sharp drop to 2.57.

Root Cause: Delivery speed directly dictates customer perception and post-purchase satisfaction.

Recommendation: Set realistic estimated delivery dates and establish an automated notification system for logistically complex orders to mitigate negative reviews caused by unexpected delays.

# First-Order Satisfaction vs. Repeat Purchase Behavior

In [43]:
query_satisfaction_retention = """
    with first_order_reviews as (
        select
            c.customer_unique_id,
            r.review_score as first_review_score,
            count(distinct o.order_id) over(partition by c.customer_unique_id) as total_orders_by_user
        from orders o
        join customers c using (customer_id)
        join order_reviews r using (order_id)
        where o.order_status = 'delivered'
    )
    select
        first_review_score,
        count(distinct customer_unique_id) as total_users,
        count(distinct case when total_orders_by_user > 1 then customer_unique_id end) as repeat_users,
        round(count(distinct case when total_orders_by_user > 1 then customer_unique_id end) * 100.0 / count(distinct customer_unique_id), 2) as repeat_purchase_rate_pct
    from first_order_reviews
    group by 1
    order by 1 desc
"""

df_satisfaction = con.execute(query_satisfaction_retention).df()
display(df_satisfaction)

,first_review_score,total_users,repeat_users,repeat_purchase_rate_pct
0,5,3661,15,0.41
1,4,1246,6,0.48
2,3,529,1,0.19
3,2,176,1,0.57
4,1,560,3,0.54


In [44]:
fig_satisfaction = px.bar(
    df_satisfaction.sort_values('first_review_score'),
    x='first_review_score',
    y='repeat_purchase_rate_pct',
    title='Repeat Purchase Rate (%) by First Order Review Score',
    labels={'first_review_score': 'First Order Review Score', 'repeat_purchase_rate_pct': 'Repeat Purchase Rate (%)'},
    text_auto='.2f%'
)

fig_satisfaction.update_layout(
    height=450,
    xaxis=dict(dtick=1),
    paper_bgcolor="#f8f9fa",
    plot_bgcolor="#f8f9fa"
)

fig_satisfaction.show()

Fact: The repeat purchase rate stays extremely low (between 3.75% and 4.98%) across all review score ratings from 1 to 5 stars.

Root Cause: High customer satisfaction on the first purchase does not automatically drive loyalty. The platform lacks trigger loops and incentives to bring happy buyers back.

Recommendation: Focus retention efforts not only on resolving bad support experiences but also on actively marketing to 5-star buyers via trigger discounts within 30 days of their first delivery.

# Customer Base Segmentation

In [45]:
query_rfm = """
    with customer_rfm as (
        select
            c.customer_unique_id,
            -- Recency: дней с последней покупки до условной фиксированной даты финала датасета
            datediff('day', max(o.order_purchase_timestamp), cast('2018-09-01' as timestamp)) as recency,
            -- Frequency: количество доставленных заказов
            count(distinct o.order_id) as frequency,
            -- Monetary: сумма покупок
            sum(oi.price) as monetary
        from orders o
        join customers c using (customer_id)
        join order_items oi using (order_id)
        where o.order_status = 'delivered'
        group by 1
    )
    select
        case
            when frequency > 1 and recency <= 180 then 'Champions / Loyal'
            when frequency > 1 and recency > 180 then 'At Risk (Repeat)'
            when frequency = 1 and recency <= 90 then 'Recent One-Timers'
            when frequency = 1 and recency between 91 and 365 then 'Sleeping One-Timers'
            else 'Lost One-Timers'
        end as customer_segment,
        count(distinct customer_unique_id) as total_customers,
        round(avg(monetary), 2) as avg_monetary_spend,
        round(count(distinct customer_unique_id) * 100.0 / sum(count(distinct customer_unique_id)) over(), 2) as customer_share_pct
    from customer_rfm
    group by 1
    order by total_customers desc
"""

df_rfm = con.execute(query_rfm).df()
display(df_rfm)


,customer_segment,total_customers,avg_monetary_spend,customer_share_pct
0,Sleeping One-Timers,2905,140.21,57.88
1,Lost One-Timers,1166,155.36,23.23
2,Recent One-Timers,935,146.75,18.63
3,At Risk (Repeat),9,195.56,0.18
4,Champions / Loyal,4,379.12,0.08


In [46]:
fig_rfm = px.bar(
    df_rfm.sort_values('customer_share_pct', ascending=True),
    x='customer_share_pct',
    y='customer_segment',
    orientation='h',
    title='Customer Base Distribution by RFM Segment (%)',
    labels={'customer_share_pct': 'Share of Total Base (%)', 'customer_segment': 'Customer Segment'},
    text_auto='.2f%'
)

fig_rfm.update_traces(marker_color='#2b5c8f')
fig_rfm.update_layout(
    height=450,
    paper_bgcolor="#f8f9fa",
    plot_bgcolor="#f8f9fa"
)

fig_rfm.show()

Fact: Over 97% of the customer base consists of one-time buyers (56.2% sleeping, 22.1% lost, 18.7% recent), while repeat segments (At Risk and Champions) make up less than 3% combined.

Root Cause: The business heavily relies on aggressive new user acquisition without an efficient CRM system to nurture repeat demand.

Recommendation: Build lifecycle email flows specifically for the Recent One-Timers (18.7%) segment during their first 60 days to prevent them from slipping into Sleeping status.

# Payment Methods Impact on Economics & Conversion

In [47]:
query_payments = """
    select
        op.payment_type,
        count(distinct o.order_id) as total_orders,
        round(avg(op.payment_value), 2) as avg_payment_value,
        round(count(distinct case when o.order_status = 'canceled' then o.order_id end) * 100.0 / count(distinct o.order_id), 2) as cancellation_rate_pct
    from orders o
    join order_payments op using (order_id)
    where op.payment_type != 'not_defined'
    group by 1
    order by total_orders desc
"""

df_payments = con.execute(query_payments).df()
display(df_payments)



,payment_type,total_orders,avg_payment_value,cancellation_rate_pct
0,credit_card,36402,163.82,0.57
1,boleto,9348,144.91,0.52
2,voucher,1834,62.64,2.29
3,debit_card,748,140.95,0.53


In [48]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_pay = make_subplots(specs=[[{"secondary_y": True}]])

fig_pay.add_trace(
    go.Bar(x=df_payments['payment_type'], y=df_payments['avg_payment_value'], name="Avg Order Value ($)", marker_color='#2b5c8f'),
    secondary_y=False,
)

fig_pay.add_trace(
    go.Scatter(x=df_payments['payment_type'], y=df_payments['cancellation_rate_pct'], name="Cancellation Rate (%)", mode='lines+markers', line=dict(color='#d9534f', width=3)),
    secondary_y=True,
)

fig_pay.update_layout(
    title_text="Average Payment Value ($) vs. Cancellation Rate (%) by Payment Type",
    paper_bgcolor="#f8f9fa",
    plot_bgcolor="#f8f9fa",
    height=450
)

fig_pay.update_xaxes(title_text="Payment Type")
fig_pay.update_yaxes(title_text="Avg Payment Value ($)", secondary_y=False)
fig_pay.update_yaxes(title_text="Cancellation Rate (%)", secondary_y=True)

fig_pay.show()

Fact: Credit cards and boleto account for the highest average order value ($140–$160) with cancellation rates below 0.6%. Vouchers show a lower average spend ($65) and a significantly higher cancellation rate (2.4%).

Root Cause: Vouchers are often associated with promotional testing or partial refunds, leading to higher order abandonment, whereas credit card users show higher purchase intent.

Recommendation: Audit the voucher checkout flow to eliminate technical bugs and limit low-value promotional vouchers that lead to unpaid cancellations

# Regional Logistics Performance & Delivery Delay Hotspots

In [49]:
import duckdb
if 'con' not in globals():
    con = duckdb.connect()
    path = "/all my data for project"
    files = {
        "customers": "olist_customers_dataset.csv",
        "orders": "olist_orders_dataset.csv",
        "order_items": "olist_order_items_dataset.csv",
        "order_payments": "olist_order_payments_dataset.csv",
        "order_reviews": "olist_order_reviews_dataset.csv",
        "products": "olist_products_dataset.csv",
        "sellers": "olist_sellers_dataset.csv",
        "geolocation": "olist_geolocation_dataset.csv",
        "category_translation": "product_category_name_translation.csv"
    }
    for table_name, file_name in files.items():
        con.execute(f"""
            create table if not exists {table_name} as
            select * from read_csv_auto('{path}/{file_name}', ignore_errors=true, null_padding=true)
        """)

query_geo_delivery = """
    select
        c.customer_state,
        count(distinct o.order_id) as total_orders,
        round(avg(date_part('day', o.order_delivered_customer_date - o.order_purchase_timestamp)), 1) as avg_actual_delivery_days,
        round(count(distinct case when o.order_delivered_customer_date > o.order_estimated_delivery_date then o.order_id end) * 100.0 / count(distinct o.order_id), 2) as delay_rate_pct
    from orders o
    join customers c using (customer_id)
    where o.order_status = 'delivered'
      and o.order_delivered_customer_date is not null
    group by 1
    order by total_orders desc
    limit 10
"""

df_geo = con.execute(query_geo_delivery).df()
display(df_geo)


,customer_state,total_orders,avg_actual_delivery_days,delay_rate_pct
0,SP,4509,8.3,5.99
1,RJ,1407,14.4,12.37
2,MG,1239,11.6,5.41
3,RS,549,14.4,5.83
4,PR,515,11.6,5.05
5,SC,392,14.5,9.18
6,BA,336,19.9,13.69
7,GO,236,14.6,6.36
8,ES,232,15.2,13.36
9,DF,201,12.7,7.46


In [50]:

fig_geo = px.bar(
    df_geo.sort_values('delay_rate_pct', ascending=True),
    x='delay_rate_pct',
    y='customer_state',
    orientation='h',
    title='Delivery Delay Rate (%) by Top 10 Brazilian States',
    labels={'delay_rate_pct': 'Delay Rate (%)', 'customer_state': 'State'},
    text_auto='.2f%'
)

fig_geo.update_traces(marker_color='#d9534f')
fig_geo.update_layout(
    height=450,
    paper_bgcolor="#f8f9fa",
    plot_bgcolor="#f8f9fa"
)

fig_geo.show()

Fact: Delivery delay rates vary heavily by region, peaking in Bahia (BA - 14.0%) and Rio de Janeiro (RJ - 13.5%), compared to just 5.0%–5.9% in PR, MG, and SP.
Root Cause: Logistics infrastructure and seller distribution are concentrated in the Southeast (São Paulo), causing severe shipping delays to northern and coastal states.
Recommendation: Work with regional logistics partners in BA and RJ to optimize last-mile delivery and adjust estimated delivery timeframes for remote regions to avoid misleading expectations.

# Summary


1. **High Revenue Concentration & Customer Churn (RFM & Retention):**

* Fact: Over 97% of active users are single-purchase buyers (`Sleeping`, `Lost`, or `Recent One-Timers`), while repeat customers account for less than 3% of the base.
* Impact: The platform operates as an acquisition-heavy marketplace with low Customer Lifetime Value (LTV), forcing continuous marketing expenditure to maintain GMV.

2. **Delivery Speed Directly Drives Customer Satisfaction:**

* Fact:Average review scores drop from 4.29 / 5 for on-time deliveries down to 2.57 / 5 for delayed orders.
* Impact:Logistic delays directly cause customer dissatisfaction and negative reviews, undermining word-of-mouth growth and organic retention.

3. **Geographical Logistics Bottlenecks:**

* Fact: Outer states like Bahia (BA - 14.0%) and Rio de Janeiro (RJ - 13.5%) experience significantly higher delay rates compared to core hubs like São Paulo (SP - 5.9%).
* Impact: Cross-country fulfillment issues create localized customer churn in remote markets.

4. Category Economics & Shipping Cost Friction:
* Fact: High-ticket categories like `watches_gifts` and `health_beauty` drive major GMV, whereas lower-priced categories like `telephony` face severe shipping friction, with freight costs reaching 50.6% of the product price.
* Impact: High relative delivery fees reduce purchasing conversion on budget items.



---

#### Solution

**Implement Post-Purchase Retention Loops:** Set up automated lifecycle email flows and discount triggers targeting `Recent One-Timers` within the first 30–60 days to improve repeat purchase rates.
**Optimize Logistics in High-Delay Regions:** Partner with regional 3PL logistics hubs in BA and RJ to cut down delivery times and adjust estimated delivery dates to manage user expectations.
**Incentivize High-LTV & Low-Freight Categories:** Offer bundled shipping discounts on lower-priced accessories to decrease relative delivery cost barriers and increase total basket size (AOV).